### Today we are going to perform the simple classification of the amazon reviews' sentiment.

### Please, download the dataset amazon_baby.csv.

Here we are importing libraries to work with data, and also i have changed a little bit remove_punctation:
  - it checks if text is string
  - if it is it creates a dictionary translator which deletes all the interpunction.
  - if its not it just returns ""

  We prepare data here to analise it later.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string
from sklearn.linear_model import LogisticRegression

def remove_punctuation(text):
    import string
    if isinstance(text, str):
      translator = str.maketrans('', '', string.punctuation)
      return text.translate(translator)
    else:
        return ""

baby_df = pd.read_csv('amazon_baby.csv')
baby_df.head()

,name,review,rating
0,Planetwise Flannel Wipes,"These flannel wipes are OK, but in my opinion ...",3
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5


As we can see the dataset contains customer reviews of baby products on Amazon. It has three columns:
 - name (name of the product)
 - review (text of a customer review)
 - rating - numerical rating given by customer in range (1 - 5)

## Exercise 1 (data preparation)
a) Remove punctuation from reviews using the given function.   
b) Replace all missing (nan) revies with empty "" string.  
c) Drop all the entries with rating = 3, as they have neutral sentiment.   
d) Set all positive ($\geq$4) ratings to 1 and negative($\leq$2) to -1.

In excercise 1 we are supposed to prepare data even more, by creating a new column "review_clean" which has no punctuation.

In [26]:
#a)
baby_df['review_clean'] = baby_df['review'].apply(remove_punctuation)

baby_df.head()

print(baby_df['rating'].value_counts())
#short test:
#baby_df["review"][4] == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'
#remove_punctuation(baby_df["review"][4]) == 'All of my kids have cried nonstop when I tried to ween them off their pacifier until I found Thumbuddy To Loves Binky Fairy Puppet  It is an easy way to work with your kids to allow them to understand where their pacifier is going and help them part from itThis is a must buy book and a great gift for expecting parents  You will save them soo many headachesThanks for this book  You all rock'

rating
5    107054
4     33205
3     16779
1     15183
2     11310
Name: count, dtype: int64


I have displayed the values because i needed it in analysis in c) :)




-----------
In b) excercise i have handled the missing values in the review column.

I checked if the review has any NaN`s. Then replaced any missing values with "".

This ensures that rows have a string in review, which is important for processing.

In [27]:
#b)
baby_df['review'] = baby_df['review'].fillna("")
#short test:
baby_df["review"][38] == baby_df["review"][38]
baby_df.head()

,name,review,rating,review_clean
0,Planetwise Flannel Wipes,"These flannel wipes are OK, but in my opinion ...",3,These flannel wipes are OK but in my opinion n...
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5,it came early and was not disappointed i love ...
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5,Very soft and comfortable and warmer than it l...
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5,This is a product well worth the purchase I h...
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5,All of my kids have cried nonstop when I tried...


Basically here we have deleted the reviews with rating 3 which are the least important. Why? Because they are most neutral. Often in such an analysis these are removed to have some kind of binary classification like in excercise d) we do.

In [28]:
#c)
#short test:
print("Before dropping: ", sum(baby_df["rating"] == 3))

baby_df = baby_df[baby_df['rating'] != 3]

print("After dropping: ", sum(baby_df["rating"] == 3))


baby_df.head()

Before dropping:  16779
After dropping:  0


,name,review,rating,review_clean
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,5,it came early and was not disappointed i love ...
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,5,Very soft and comfortable and warmer than it l...
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,5,This is a product well worth the purchase I h...
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,5,All of my kids have cried nonstop when I tried...
5,Stop Pacifier Sucking without tears with Thumb...,"When the Binky Fairy came to our house, we did...",5,When the Binky Fairy came to our house we didn...


As we said before we changed the positive ratings such as (4, 5) to 1 and negative like a (1, 2) to -1. This is standard for preprocessing for binary classification tasks.

In [29]:
#d)
baby_df['rating'] = np.where(baby_df['rating'] >= 4, 1, -1)


print(baby_df['rating'].value_counts())

baby_df.head()

rating
 1    140259
-1     26493
Name: count, dtype: int64


,name,review,rating,review_clean
1,Planetwise Wipe Pouch,it came early and was not disappointed. i love...,1,it came early and was not disappointed i love ...
2,Annas Dream Full Quilt with 2 Shams,Very soft and comfortable and warmer than it l...,1,Very soft and comfortable and warmer than it l...
3,Stop Pacifier Sucking without tears with Thumb...,This is a product well worth the purchase. I ...,1,This is a product well worth the purchase I h...
4,Stop Pacifier Sucking without tears with Thumb...,All of my kids have cried non-stop when I trie...,1,All of my kids have cried nonstop when I tried...
5,Stop Pacifier Sucking without tears with Thumb...,"When the Binky Fairy came to our house, we did...",1,When the Binky Fairy came to our house we didn...


## CountVectorizer
In order to analyze strings, we need to assign them numerical values. We will use one of the simplest string representation, which transforms strings into the $n$ dimensional vectors. The number of dimensions will be the size of our dictionary, and then the values of the vector will represent the number of appereances of the given word in the sentence.

In [31]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
reviews_train_example = ["We like apples",
                   "We hate oranges",
                   "I adore bananas",
                   "We like like apples and oranges",
                   "They dislike bananas"]

X_train_example = vectorizer.fit_transform(reviews_train_example)

print(vectorizer.get_feature_names_out())
print(X_train_example.todense())



['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 1 0 1]
 [1 0 0 1 0 0 0 0 0 0]
 [0 1 1 0 0 0 2 1 0 1]
 [0 0 0 1 1 0 0 0 1 0]]


What is important to notice here is that countVectorizes converts text in something like a bag-of-words representation.

We create the vectorizer, and some example of training data.

fit_transofrm does two things:
  - fit -lears the vocab
  - transform - converts each text into sparse matrix of word counts.

result X_train_example is a 5 x n matrix.

We can see that the output is in alfabethical order and the 1 and 0 are representing the words that came up in the text.


In [33]:
reviews_test_example = ["They like bananas",
                   "We hate oranges bananas and apples",
                   "We love bananas"] #New word!

X_test_example = vectorizer.transform(reviews_test_example)

print(vectorizer.get_feature_names_out())
print(X_test_example.todense())

['adore' 'and' 'apples' 'bananas' 'dislike' 'hate' 'like' 'oranges' 'they'
 'we']
[[0 0 0 1 0 0 1 0 1 0]
 [0 1 1 1 0 1 0 1 0 1]
 [0 0 0 1 0 0 0 0 0 1]]


We should acknowledge few facts. Firstly, CountVectorizer does not take order into account. Secondly, it ignores one-letter words (this can be changed during initialization). Finally, for test values, CountVectorizer ignores words which are not in it's dictionary.

## Exercise 2
a) Split dataset into training and test sets.     
b) Transform reviews into vectors using CountVectorizer.

In [37]:
#a)
from sklearn.model_selection import train_test_split
X = baby_df['review_clean']   # teksty
y = baby_df['rating']         # etykiety (-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

#it divides the date 80%/20%

Train size: 133401
Test size: 33351


Now we move on to the original dataset. First we are splitting data into 80% / 20%.

We can see the training and test count in the console.

In [38]:
#b)
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

# Uczymy vectorizera na zbiorze treningowym
X_train_vectorized = vectorizer.fit_transform(X_train)

# Wektor transformacji testu — NIE używamy .fit!
X_test_vectorized = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Train matrix shape:", X_train_vectorized.shape)
print("Test matrix shape:", X_test_vectorized.shape)

Vocabulary size: 121805
Train matrix shape: (133401, 121805)
Test matrix shape: (33351, 121805)


Here again:
  - fit (learns the vocab from training set)
  - transform (COnverts the training texts into sparse matrix)

Why only use transform on test set?
We want to use the same vocab learned from training to ensure new features. New words are ignored.



## Exercise 3
a) Train LogisticRegression model on training data (reviews processed with CountVectorizer, ratings as they were).   
b) Print 10 most positive and 10 most negative words.

In [39]:
#a)
model = LogisticRegression()

model.fit(X_train_vectorized, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

X_train... - the bag of words

y_train - labels (1 , -1)

In [41]:
#b)
words = vectorizer.get_feature_names_out()
coeffs = model.coef_[0]

sorted_indices = np.argsort(coeffs)

print("Most negative words: ")
print(words[sorted_indices[:10]])

print("\nMost positive words: ")
print(words[sorted_indices[-10:]])
#hint: model.coef_, vectorizer.get_feature_names()

Most negative words: 
['worst' 'disappointing' 'useless' 'poorly' 'poor' 'terrible' 'concept'
 'returning' 'returned' 'horrible']

Most positive words: 
['exactly' 'highly' 'complaint' 'satisfied' 'glad' 'amazing' 'worry'
 'pleased' 'awesome' 'excellent']


As we can see in the print the negative words have strong negative coefficients, meaning their present in a review strongly predicts a negative statement.

Same case is with positive their presence increases a chance that the review was positive.

## Exercise 4
a) Predict the sentiment of test data reviews.   
b) Predict the sentiment of test data reviews in terms of probability.   
c) Find five most positive and most negative reviews.   
d) Calculate the accuracy of predictions.

In [43]:
#a)

y_pred = model.predict(X_test_vectorized)

print(y_pred[:10])

[ 1 -1 -1  1  1  1  1  1  1 -1]


1 means that the model predicts the review is positive

-1 means that was negative

This is simply a models classification output for reviews in test set.

In [44]:
#b)
y_proba = model.predict_proba(X_test_vectorized)

print(y_proba[:5])
#hint: model.predict_proba()

[[0.39808426 0.60191574]
 [0.79754664 0.20245336]
 [0.70249309 0.29750691]
 [0.00154467 0.99845533]
 [0.00401972 0.99598028]]


predict_proba() returns the model`s confidence for each review for each review: probability it is negative vs positive

usefull for ranking reviews by sentiment strength

In [47]:
#c)

sorted_indices = np.argsort(y_proba[:, 1])

most_negative_idx = sorted_indices[:5]

most_positive_idx = sorted_indices[-5:]

print("Most positive reviews: ")
for i in most_positive_idx[::-1]:
  print(f"Prob={y_proba[i, 1]:.2f} | Review: {X_test.iloc[i]}\n")

for i in most_negative_idx[::-1]:
  print(f"Prob={y_proba[i, 1]:.2f} | Review: {X_test.iloc[i]}\n")


#hint: use the results of b)

Most positive reviews: 
Prob=1.00 | Review: My husband and I cannot state enough how much we value and appreciate this swing No amount of money could have provided to us and our baby what this swing has offeredThis swing has brought peace quiet and happiness to our household Our newborn baby girl was a bit colicky when she was first born for the first 8 weeks She would sleep well through the night but would be fussy and not willing to sleepnap during the day She wanted to be rocked to sleep which left our arms very tired and sore Well this swing to the rescue It puts our little girl to sleep in less than 10 minutes but not before it thoroughly entertains her with its mirrored surface that the baby faces She loves it and coos at it all the time its wonderful mobile that she loves and even the sounds which she very much appreciates I also love how the direction of swinging can be changes from front to back or side to side Our daughter did not care for the side to side motion but LOVED ba

Most positive reviews (Prob around 1.00)
  - These reviews have the highest predicted probability of being positive
  - The model is almost certain these reviews express strong satisfaction

Most negative reviews (Prob around 0.00)
  - These reviews have the lowest predicted probability of being positive

Reviews that are long still can be classified correctly as long as the overall sentiment is consistent.

In [49]:
#d)
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_vectorized)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy on test set: ", accuracy)

Accuracy on test set:  0.9336151839525052


Out of all the reviews in my test set about 93% were correctly classified as positive or neative in this model.

Very strong performance.

## Exercise 5
In this exercise we will limit the dictionary of CountVectorizer to the set of significant words, defined below.


a) Redo exercises 2-5 using limited dictionary.   
b) Check the impact of all the words from the dictionary.   
c) Compare accuracy of predictions and the time of evaluation.

In [50]:
significant_words = ['love','great','easy','old','little','perfect','loves','well','able','car','broke','less','even','waste','disappointed','work','product','money','would','return']

In [53]:
#a)
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_limited = CountVectorizer(vocabulary=significant_words)

X_train_vectorized_limited = vectorizer_limited.fit_transform(X_train)
X_test_vectorized_limited = vectorizer_limited.transform(X_test)

print("Vocabulary size:", len(vectorizer_limited.get_feature_names_out()))
print("Train matrix shape:", X_train_vectorized_limited.shape)
print("Test matrix shape:", X_test_vectorized_limited.shape)

Vocabulary size: 20
Train matrix shape: (133401, 20)
Test matrix shape: (33351, 20)


Here we restricted the vocabulary to predefined set of significant words.

This means the model will only consider these 20 words when converting text into a numerical matrix.

All other words are ignored, which simplifies the space and focuses the model on words that are meaningfull.

In [54]:
from sklearn.linear_model import LogisticRegression
import time

model_limited = LogisticRegression(max_iter=1000)

# Measure training time
start_time = time.time()
model_limited.fit(X_train_vectorized_limited, y_train)
train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")

Training completed in 0.26 seconds


Created a Logistic Regression classifier and trained it on the vectorized training data (using only the significant words).

Measured the training time.


In [55]:
coefficients = model_limited.coef_[0]

# Most positive
top_positive_idx = coefficients.argsort()[-10:][::-1]
top_positive_words = np.array(significant_words)[top_positive_idx]

# Most negative
top_negative_idx = coefficients.argsort()[:10]
top_negative_words = np.array(significant_words)[top_negative_idx]

print("Most positive words:", top_positive_words)
print("Most negative words:", top_negative_words)


Most positive words: ['loves' 'perfect' 'love' 'easy' 'great' 'little' 'well' 'able' 'car'
 'old']
Most negative words: ['disappointed' 'return' 'waste' 'broke' 'money' 'work' 'even' 'would'
 'product' 'less']


Looked at the model coefficients for each word.

Sorted words by coefficient values:

Top positive → words strongly associated with positive reviews.

Top negative → words strongly associated with negative reviews.

In [56]:
# Predictions
y_pred_limited = model_limited.predict(X_test_vectorized_limited)

# Probabilities
y_proba_limited = model_limited.predict_proba(X_test_vectorized_limited)[:,1]  # positive class


Used the trained model to predict sentiment on the test set.

Also predicted probabilities that each review is positive (predict_proba).

In [57]:
sorted_indices_limited = np.argsort(y_proba_limited)

most_negative_idx = sorted_indices_limited[:5]
most_positive_idx = sorted_indices_limited[-5:]

print("Most positive reviews (limited dictionary):")
for i in most_positive_idx[::-1]:
    print(f"Prob={y_proba_limited[i]:.2f} | Review: {X_test.iloc[i]}\n")

print("Most negative reviews (limited dictionary):")
for i in most_negative_idx:
    print(f"Prob={y_proba_limited[i]:.2f} | Review: {X_test.iloc[i]}\n")


Most positive reviews (limited dictionary):
Prob=1.00 | Review: We bought this stroller after selling our beloved BOB rev on craigslist We used the BOB for 9 months for my son but it just wasnt practical I dont jogrun it didnt have a big basket and was very bulky to take into stores quickly However I did love how it unfolded easily but it was heavy to fold up and lift into my small trunk myself Overall I didnt realize what Id need in a stroller until AFTER I had my son Live  learn We did love how easily the BOB would go over pretty much anything Nevertheless we sold it and after extensive research on strollers we decided it was between the uppababy brand because of the large baskets OR the city mini GT because of its easy fold up design After looking over both strollers I decided on the uppababy cruz because of a few main factors It SITS UP I cant tell you how much my son hates being reclined when he is just riding in the stroller and not napping The BOB and the City Mini had a slight 

Sorted the test reviews by the predicted probability of positive sentiment.

Took:

Top 5 reviews with highest probability → most positive.

Bottom 5 reviews with lowest probability → most negative.

In [59]:
from sklearn.metrics import accuracy_score
accuracy_limited = accuracy_score(y_test, y_pred_limited)
print("Accuracy with limited dictionary:", accuracy_limited)


Accuracy with limited dictionary: 0.8689994303019399


Measure how well the model performs on unseen data.

Accuracy of 86%.

Accuracy: Compare accuracy (all words) vs accuracy_limited (significant words).

Evaluation time: Compare time to train & predict.

Limiting vocabulary usually:

Reduces dimensionality → faster training/prediction.

May slightly reduce accuracy because some informative words are ignored.